批量生成lobsterin文件并且结合lobster进行cohp计算的脚本

In [3]:
import os
import shutil
from pymatgen.core import Structure
from pymatgen.core.periodic_table import Element


In [4]:
def is_transition_metal(element):
    try:
        return Element(element).is_transition_metal
    except:
        return False


In [9]:
structure = Structure.from_file("44.vasp")
transition_metals = [site for site in structure if is_transition_metal(site.specie)]
tm_o_dict = {}

# 遍历结构，找到过渡金属
for tm_index, tm_site in enumerate(structure):
    if not is_transition_metal(tm_site.specie):
        continue

    # 找到邻居 (site, distance, index)
    neighbors = structure.get_neighbors(tm_site, r=3.0)
    # 只取氧
    oxygen_neighbors = [(n[0], n[1], n[2]) for n in neighbors if n[0].specie.symbol == "O"]
    # 按距离排序
    oxygen_neighbors_sorted = sorted(oxygen_neighbors, key=lambda x: x[1])
    # 取前6个 O 的 index
    o_indices = [o[2] for o in oxygen_neighbors_sorted[:6]]

    # 保存到字典
    tm_o_dict[tm_index] = o_indices

# 打印结果
# for tm_idx, o_list in tm_o_dict.items():
#     print(f"TM index {tm_idx}: O indices {o_list}")
print(tm_o_dict)

{36: [84, 114, 64, 104, 74, 94], 37: [95, 105, 75, 115, 65, 85], 38: [106, 66, 86, 116, 96, 76], 39: [117, 107, 67, 77, 87, 97], 40: [98, 108, 78, 88, 68, 118], 41: [89, 119, 69, 109, 99, 79], 42: [81, 111, 102, 90, 62, 70], 43: [63, 71, 91, 103, 80, 110], 44: [72, 113, 83, 64, 92, 104], 45: [112, 73, 65, 82, 105, 93], 46: [66, 106, 115, 85, 94, 74], 47: [67, 84, 107, 95, 114, 75], 48: [117, 87, 108, 68, 96, 76], 49: [77, 69, 97, 86, 109, 116], 50: [101, 91, 118, 60, 80, 78], 51: [61, 79, 81, 90, 100, 119], 52: [93, 103, 110, 82, 70, 62], 53: [71, 63, 83, 111, 92, 102], 54: [100, 60, 80, 90, 70, 110], 55: [112, 72, 92, 82, 102, 62], 56: [89, 78, 98, 119, 60, 100], 57: [61, 118, 79, 101, 88, 99], 58: [73, 113, 104, 65, 94, 85], 59: [99, 88, 68, 109, 76, 116]}


In [1]:
# 生成lobsterin的函数
def lobster_in_generator(cohp_pair, energy_min=-10, energy_max=10):
    """
    根据输入输出lobsterin文件
    :param energy_min: int，能量下界
    :param energy_max: int，能量上界
    :param cohp_pair: tuple，给定原子对 (1, 2)
    :return: lobsterin list, 一个列表包含不同的字符串，用于lobsterin的生成
    """
    min_str = "COHPstartEnergy  {0}".format(energy_min)
    max_str = "COHPendEnergy    {0}".format(energy_max)
    pair_str = "cohpbetween atom {0} and atom {1}".format(cohp_pair[0], cohp_pair[1])
    basis_str = "basisSet pbeVaspFit2015"
    lobster_in_list = [min_str, max_str, basis_str, pair_str]
    return lobster_in_list


['COHPstartEnergy  -10',
 'COHPendEnergy    10',
 'basisSet pbeVaspFit2015',
 'cohpbetween atom 2 and atom 112']

In [14]:
current_dir = os.getcwd()
for key, values in tm_o_dict.items():
    for value in values:
        subdir = str(key) + "_" + str(value)
        os.mkdir(subdir)
        work_path = os.path.join(current_dir, subdir) # 计算的路径

        lobster_in_path = os.path.join(work_path, "lobsterin")  # lobsterin的路径
        lobsterin = lobster_in_generator((key, value))  # 此次循环中的lobsterin

        files_to_copy = ["POSCAR", "POTCAR", "CHGCAR", "OUTCAR", "CONTCAR", "KPOINTS", "WAVECAR", "lobster.sh", "vasprun.xml"]

        for file in files_to_copy:
            shutil.copy(file, work_path)
        with open(lobster_in_path, "w") as f:
            for line in lobsterin:
                f.write(line + "\n")




36 84
36 114
36 64
36 104
36 74
36 94
37 95
37 105
37 75
37 115
37 65
37 85
38 106
38 66
38 86
38 116
38 96
38 76
39 117
39 107
39 67
39 77
39 87
39 97
40 98
40 108
40 78
40 88
40 68
40 118
41 89
41 119
41 69
41 109
41 99
41 79
42 81
42 111
42 102
42 90
42 62
42 70
43 63
43 71
43 91
43 103
43 80
43 110
44 72
44 113
44 83
44 64
44 92
44 104
45 112
45 73
45 65
45 82
45 105
45 93
46 66
46 106
46 115
46 85
46 94
46 74
47 67
47 84
47 107
47 95
47 114
47 75
48 117
48 87
48 108
48 68
48 96
48 76
49 77
49 69
49 97
49 86
49 109
49 116
50 101
50 91
50 118
50 60
50 80
50 78
51 61
51 79
51 81
51 90
51 100
51 119
52 93
52 103
52 110
52 82
52 70
52 62
53 71
53 63
53 83
53 111
53 92
53 102
54 100
54 60
54 80
54 90
54 70
54 110
55 112
55 72
55 92
55 82
55 102
55 62
56 89
56 78
56 98
56 119
56 60
56 100
57 61
57 118
57 79
57 101
57 88
57 99
58 73
58 113
58 104
58 65
58 94
58 85
59 99
59 88
59 68
59 109
59 76
59 116
